In [ ]:
# Dia 4 - Parte 1: engenharia de features pro clustering
#
# Decisoes tomadas:
# - Perfil combinado: 3 features por municipio (nao por vacina), juntando as
#   5 vacinas num comportamento geral.
# - Municipio precisa de pelo menos 7 dos 10 anos (2014-2023) com dado valido
#   pra entrar no clustering. Quem nao atingir isso fica de fora (documentado,
#   nao e removido dos dados brutos, so nao participa do modelo).
#
# Passo 1: carregar a serie temporal, usando cobertura_capada (nao a bruta)

import numpy as np
import pandas as pd

df = pd.read_csv(
    "../data/processed/serie_temporal_mvp.csv",
    dtype={"ibge6": str, "uf": str, "imunizante": str},
)

print(f"Linhas carregadas: {len(df)}")
print(f"Colunas: {list(df.columns)}")

In [ ]:
# Passo 2: media de cobertura_capada entre as 5 vacinas, por (ibge6, ano),
# ignorando NaN -- "cobertura geral do municipio naquele ano"

cobertura_geral = df.groupby(["ibge6", "ano"])["cobertura_capada"].mean().reset_index()

print(f"cobertura_geral shape: {cobertura_geral.shape}")
print(f"Linhas com cobertura geral nula (nenhuma das 5 vacinas tinha dado): {cobertura_geral['cobertura_capada'].isna().sum()}")
cobertura_geral.head()

In [ ]:
# Passo 3: contar quantos dos 10 anos cada municipio tem valor nao-nulo na
# serie geral, e filtrar quem tem >= 7 anos validos

anos_validos_por_municipio = (
    cobertura_geral.dropna(subset=["cobertura_capada"])
    .groupby("ibge6")["ano"].nunique()
)

todos_ibge6 = sorted(df["ibge6"].unique())
anos_validos_full = pd.Series(0, index=todos_ibge6, name="n_anos_validos")
anos_validos_full.update(anos_validos_por_municipio)

municipios_ok = anos_validos_full[anos_validos_full >= 7].index
municipios_fora = anos_validos_full[anos_validos_full < 7].index

print(f"Municipios totais: {len(todos_ibge6)}")
print(f"Passaram no filtro (>= 7 anos validos): {len(municipios_ok)}")
print(f"Ficaram de fora (< 7 anos validos): {len(municipios_fora)}")
print()
print("Distribuicao de n_anos_validos (todos os municipios):")
print(anos_validos_full.value_counts().sort_index())

if len(municipios_fora) > 0:
    ibge6_para_uf = df[["ibge6", "uf"]].drop_duplicates().set_index("ibge6")["uf"]
    print()
    print("UF dos municipios que ficaram de fora:")
    print(ibge6_para_uf.loc[municipios_fora].value_counts())

In [ ]:
# Passo 4: pros municipios que passaram no filtro, calcular as 3 features a
# partir da serie de anos validos (so os anos com dado, media/desvio/slope
# calculados so sobre eles):
# - media_cobertura: media dos valores anuais
# - variabilidade_cobertura: desvio padrao dos valores anuais
# - tendencia_cobertura: slope de uma regressao linear simples (valor ~ ano),
#   via numpy.polyfit grau 1

def calcula_features(grupo):
    serie = grupo.dropna(subset=["cobertura_capada"]).sort_values("ano")
    anos = serie["ano"].values.astype(float)
    valores = serie["cobertura_capada"].values
    slope = np.polyfit(anos, valores, 1)[0]
    return pd.Series({
        "n_anos_validos": len(valores),
        "media_cobertura": valores.mean(),
        "variabilidade_cobertura": valores.std(ddof=1),
        "tendencia_cobertura": slope,
    })

cobertura_geral_ok = cobertura_geral[cobertura_geral["ibge6"].isin(municipios_ok)]
features = (
    cobertura_geral_ok.groupby("ibge6")
    .apply(calcula_features, include_groups=False)
    .reset_index()
)

print(f"Features calculadas para {len(features)} municipios")
features.head()

In [ ]:
# Passo 5: tabela final (ibge6, uf, n_anos_validos, media_cobertura,
# variabilidade_cobertura, tendencia_cobertura) e salvar em
# data/processed/features_clustering_mvp.csv

ibge6_para_uf = df[["ibge6", "uf"]].drop_duplicates().set_index("ibge6")["uf"]
features["uf"] = features["ibge6"].map(ibge6_para_uf)

features = features[[
    "ibge6", "uf", "n_anos_validos",
    "media_cobertura", "variabilidade_cobertura", "tendencia_cobertura",
]]

caminho_features = "../data/processed/features_clustering_mvp.csv"
features.to_csv(caminho_features, index=False)
print(f"Salvo em: {caminho_features}")
print(f"Shape: {features.shape}")

In [ ]:
# Passo 6: validacoes - shape, describe() das 3 features (escala de cada uma
# antes da normalizacao na Parte 2) e confirmar que nao sobrou NaN

print("shape:", features.shape)
print()

print("describe() das 3 features:")
print(features[["media_cobertura", "variabilidade_cobertura", "tendencia_cobertura"]].describe())
print()

print("NaN por coluna:")
print(features.isna().sum())

In [ ]:
# Dia 4 - Parte 2: padronizacao das features
#
# IMPORTANTE: nao foi possivel usar scikit-learn nesta maquina. O scipy (que o
# sklearn usa por baixo dos panos, inclusive pro KMeans e pro silhouette_score)
# e bloqueado pelo Device Guard/Controle de Aplicativo da organizacao ao
# carregar suas DLLs compiladas (_flapack, _qhull etc) -- testado em duas
# instalacoes Python diferentes nesta maquina, mesmo erro nas duas. O numpy
# funciona normalmente. Por decisao sua, a partir daqui StandardScaler, KMeans
# e silhouette_score sao implementados do zero com numpy puro, seguindo a
# mesma matematica que o sklearn usaria.
#
# Passo 1: carregar as features e montar X

features = pd.read_csv(
    "../data/processed/features_clustering_mvp.csv",
    dtype={"ibge6": str, "uf": str},
)

colunas_features = ["media_cobertura", "variabilidade_cobertura", "tendencia_cobertura"]
X = features[colunas_features].values

print(f"X shape: {X.shape}")

In [ ]:
# Passo 2: padronizar X (equivalente ao StandardScaler.fit_transform do sklearn)
# Conceito: cada feature tem uma escala diferente (media_cobertura varia em
# dezenas, tendencia_cobertura varia perto de zero). O KMeans usa distancia
# euclidiana, entao sem padronizar a feature de maior escala dominaria a
# distancia. Padronizar = (x - media) / desvio_padrao, feature por feature,
# deixando todas com media 0 e desvio 1.

media_colunas = X.mean(axis=0)
desvio_colunas = X.std(axis=0, ddof=0)  # ddof=0 = mesmo padrao do StandardScaler

X_padronizado = (X - media_colunas) / desvio_colunas

print("Media apos padronizacao (esperado ~0):", X_padronizado.mean(axis=0))
print("Desvio apos padronizacao (esperado ~1):", X_padronizado.std(axis=0))

In [ ]:
# Dia 4 - Parte 3 (comeco): funcoes de KMeans e silhouette score, do zero
#
# Conceito KMeans: escolhe K centros, atribui cada ponto ao centro mais
# proximo (distancia euclidiana), recalcula cada centro como a media dos
# pontos atribuidos a ele, repete ate os centros pararem de se mover
# (algoritmo de Lloyd). O resultado depende de onde os centros comecam, entao
# n_init=10 roda tudo 10 vezes com pontos de partida diferentes e fica com a
# rodada de menor inertia (soma das distancias ao quadrado de cada ponto ao
# seu centro -- quanto menor, mais compactos os clusters).
# Em vez de comecar os centros em pontos aleatorios quaisquer, usamos
# k-means++: o primeiro centro e aleatorio, e cada centro seguinte e
# sorteado com probabilidade proporcional a distancia ao quadrado do centro
# mais proximo ja escolhido -- isso espalha os centros iniciais e evita que
# dois comecem colados, que e a causa mais comum do KMeans convergir mal.
#
# Conceito silhouette score: pra cada ponto, compara a distancia media aos
# pontos do seu proprio cluster (a) com a distancia media aos pontos do
# cluster vizinho mais proximo (b). O indice do ponto e (b-a)/max(a,b), entre
# -1 e 1 -- perto de 1 significa que o ponto esta bem mais perto do seu
# cluster do que de qualquer outro (boa separacao); perto de 0 significa que
# esta na fronteira entre dois clusters; negativo significa que provavelmente
# foi colocado no cluster errado. O silhouette_score e a media desse indice
# em todos os pontos.

def kmeans_plus_plus_init(X, k, rng):
    n = X.shape[0]
    centros = np.empty((k, X.shape[1]))
    centros[0] = X[rng.integers(n)]
    dist_sq = np.sum((X - centros[0]) ** 2, axis=1)
    for i in range(1, k):
        probs = dist_sq / dist_sq.sum()
        centros[i] = X[rng.choice(n, p=probs)]
        nova_dist_sq = np.sum((X - centros[i]) ** 2, axis=1)
        dist_sq = np.minimum(dist_sq, nova_dist_sq)
    return centros

def kmeans_lloyd(X, k, centros_iniciais, max_iter=300, tol=1e-4):
    centros = centros_iniciais.copy()
    for _ in range(max_iter):
        dists = ((X[:, None, :] - centros[None, :, :]) ** 2).sum(axis=2)
        labels = dists.argmin(axis=1)
        novos_centros = centros.copy()
        for j in range(k):
            membros = X[labels == j]
            if len(membros) > 0:
                novos_centros[j] = membros.mean(axis=0)
        deslocamento = np.sum((novos_centros - centros) ** 2)
        centros = novos_centros
        if deslocamento < tol:
            break
    dists_finais = ((X[:, None, :] - centros[None, :, :]) ** 2).sum(axis=2)
    labels = dists_finais.argmin(axis=1)
    inertia = dists_finais[np.arange(len(X)), labels].sum()
    return labels, centros, inertia

def kmeans_proprio(X, k, n_init=10, random_state=42, max_iter=300):
    rng_principal = np.random.default_rng(random_state)
    melhor_inertia = np.inf
    melhor_labels = melhor_centros = None
    for _ in range(n_init):
        rng = np.random.default_rng(rng_principal.integers(0, 2**32 - 1))
        centros_iniciais = kmeans_plus_plus_init(X, k, rng)
        labels, centros, inertia = kmeans_lloyd(X, k, centros_iniciais, max_iter=max_iter)
        if inertia < melhor_inertia:
            melhor_inertia, melhor_labels, melhor_centros = inertia, labels, centros
    return melhor_labels, melhor_centros, melhor_inertia

def pairwise_dist(X):
    sq_norms = np.sum(X ** 2, axis=1)
    D2 = np.maximum(sq_norms[:, None] + sq_norms[None, :] - 2 * X @ X.T, 0)
    return np.sqrt(D2)

def silhouette_proprio(D, labels, k):
    n = len(labels)
    onehot = np.zeros((n, k))
    onehot[np.arange(n), labels] = 1
    counts = onehot.sum(axis=0)
    soma_dist_cluster = D @ onehot
    media_dist_cluster = soma_dist_cluster / counts[None, :]
    own_counts = counts[labels]
    a = np.where(
        own_counts > 1,
        soma_dist_cluster[np.arange(n), labels] / np.maximum(own_counts - 1, 1),
        0.0,
    )
    media_dist_mascarada = media_dist_cluster.copy()
    media_dist_mascarada[np.arange(n), labels] = np.inf
    b = media_dist_mascarada.min(axis=1)
    s = np.where(own_counts > 1, (b - a) / np.maximum(a, b), 0.0)
    return s.mean()

In [ ]:
# Passo: rodar K de 2 a 10, guardando inertia e silhouette de cada um.
# A matriz de distancias entre todos os pares de municipios e a mesma pra
# qualquer K (so muda quais pontos pertencem a qual cluster), entao calculamos
# ela uma unica vez e reaproveitamos.

D_padronizado = pairwise_dist(X_padronizado)

resultados_k = []
for k in range(2, 11):
    labels, centros, inertia = kmeans_proprio(X_padronizado, k, n_init=10, random_state=42)
    sil = silhouette_proprio(D_padronizado, labels, k)
    resultados_k.append({"k": k, "inertia": inertia, "silhouette": sil})
    print(f"K={k}: inertia={inertia:.1f}, silhouette={sil:.4f}")

resultados_k = pd.DataFrame(resultados_k)

In [ ]:
# Dois graficos lado a lado: metodo do cotovelo (inertia) e silhouette score,
# pra ajudar a escolher K. Ainda nao rodamos o KMeans final -- isso fica pra
# quando K for decidido.

import matplotlib.pyplot as plt

COR_MARCA = "#2a78d6"
SURFACE = "#fcfcfb"
COR_GRADE = "#e1e0d9"
COR_EIXO = "#c3c2b7"

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5.5), facecolor=SURFACE)

for ax in (ax1, ax2):
    ax.set_facecolor(SURFACE)
    ax.set_xticks(resultados_k["k"])
    ax.grid(axis="y", color=COR_GRADE, linewidth=0.8, zorder=0)
    ax.set_axisbelow(True)
    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)
    for spine in ["left", "bottom"]:
        ax.spines[spine].set_color(COR_EIXO)

ax1.plot(
    resultados_k["k"], resultados_k["inertia"],
    color=COR_MARCA, linewidth=2.2, marker="o", markersize=6, solid_capstyle="round",
)
ax1.set_title("Metodo do cotovelo", fontsize=12)
ax1.set_xlabel("K (numero de clusters)", fontsize=10)
ax1.set_ylabel("Inertia", fontsize=10)

ax2.plot(
    resultados_k["k"], resultados_k["silhouette"],
    color=COR_MARCA, linewidth=2.2, marker="o", markersize=6, solid_capstyle="round",
)
ax2.set_title("Silhouette score", fontsize=12)
ax2.set_xlabel("K (numero de clusters)", fontsize=10)
ax2.set_ylabel("Silhouette score", fontsize=10)

fig.suptitle("Escolha de K para o KMeans", fontsize=13, y=1.0)
fig.tight_layout()
fig.savefig("escolha_k_clustering.png", dpi=150)
plt.show()

In [ ]:
# Analise: K sugerido pelo cotovelo (onde a queda de inertia desacelera) vs
# K com maior silhouette score

resultados_k["queda_inertia"] = resultados_k["inertia"].diff(-1)
resultados_k["queda_pct"] = 100 * resultados_k["queda_inertia"] / resultados_k["inertia"]

print("Queda de inertia de K pro K+1 (absoluta e %):")
print(resultados_k[["k", "inertia", "queda_inertia", "queda_pct"]].round(1).to_string(index=False))
print()

k_melhor_silhouette = resultados_k.loc[resultados_k["silhouette"].idxmax(), "k"]
print(f"K com maior silhouette score: {int(k_melhor_silhouette)} ({resultados_k['silhouette'].max():.4f})")

In [ ]:
# Dia 4 - Parte 4 (final): KMeans definitivo com K=3
#
# Decisao: K=3, baseada no silhouette score (0,3672 -- o maior entre K=2..10,
# com folga). O cotovelo sugeria algo entre 5 e 7, mas o silhouette mede
# diretamente o quao bem separados os clusters ficam, e prioriza simplicidade
# de interpretacao pro portfolio.
#
# Passo 1: rodar o KMeans com o mesmo random_state/inicializacao do loop de
# K=2 a 10, pra manter consistencia com os resultados ja mostrados

labels_final, centros_final, inertia_final = kmeans_proprio(
    X_padronizado, k=3, n_init=10, random_state=42
)

print(f"Inertia final (deve bater com a linha K=3 da tabela anterior): {inertia_final:.1f}")

In [ ]:
# Passo 2: anexar o cluster de cada municipio a tabela e salvar

features["cluster"] = labels_final

caminho_clusters = "../data/processed/clusters_municipios_mvp.csv"
features[[
    "ibge6", "uf", "media_cobertura", "variabilidade_cobertura",
    "tendencia_cobertura", "cluster",
]].to_csv(caminho_clusters, index=False)

print(f"Salvo em: {caminho_clusters}")
print()
print("Tamanho de cada cluster:")
print(features["cluster"].value_counts().sort_index())

In [ ]:
# Passo 3: interpretar cada cluster usando os valores ORIGINAIS (nao
# padronizados) das 3 features, mais as 5 UFs mais frequentes de cada um

for c in sorted(features["cluster"].unique()):
    sub = features[features["cluster"] == c]
    print(f"=== Cluster {c} (n={len(sub)}) ===")
    print(f"  media_cobertura:          {sub['media_cobertura'].mean():.2f}")
    print(f"  variabilidade_cobertura:  {sub['variabilidade_cobertura'].mean():.2f}")
    print(f"  tendencia_cobertura:      {sub['tendencia_cobertura'].mean():.2f}")
    print(f"  top 5 UF: {sub['uf'].value_counts().head(5).to_dict()}")
    print()

# padrao regional: onde os estados que mais apareceram no topo/fundo do
# ranking de 2023 (Dia 3) caem entre os 3 clusters
print("Distribuicao percentual por cluster, pros estados ja vistos no Dia 3:")
for uf in ["PB", "CE", "MT", "TO", "RR", "PA", "AM", "RJ", "AC"]:
    dist = features[features["uf"] == uf]["cluster"].value_counts(normalize=True).sort_index()
    dist_fmt = {int(k): f"{v*100:.1f}%" for k, v in dist.items()}
    print(f"  {uf}: {dist_fmt}")

In [ ]:
# Passo 4: nomear cada cluster com base nos numeros calculados no passo 3
# (nao escolhido antes de calcular -- a regra abaixo le as estatisticas reais
# pra decidir o nome, entao funciona nao importa qual indice (0,1,2) o KMeans
# deu pra cada grupo)

estatisticas_cluster = features.groupby("cluster")[
    ["media_cobertura", "variabilidade_cobertura", "tendencia_cobertura"]
].mean()

cluster_alta_estavel = estatisticas_cluster["media_cobertura"].idxmax()
demais = estatisticas_cluster.index.difference([cluster_alta_estavel])
cluster_recuperacao = estatisticas_cluster.loc[demais, "tendencia_cobertura"].idxmax()
cluster_queda = estatisticas_cluster.loc[demais, "tendencia_cobertura"].idxmin()

nomes_cluster = {
    cluster_alta_estavel: "Cobertura alta e estável",
    cluster_recuperacao: "Cobertura baixa, em recuperação",
    cluster_queda: "Cobertura em queda e instável",
}

for c in sorted(nomes_cluster):
    print(f"Cluster {c}: {nomes_cluster[c]}")
    print(estatisticas_cluster.loc[c].round(2).to_string())
    print()

In [ ]:
# Passo 5: grafico de dispersao - media_cobertura (x) vs tendencia_cobertura
# (y), colorido por cluster

cores_cluster = {0: "#2a78d6", 1: "#eb6834", 2: "#1baf7a"}

fig, ax = plt.subplots(figsize=(9, 7), facecolor=SURFACE)
ax.set_facecolor(SURFACE)

for c in sorted(features["cluster"].unique()):
    sub = features[features["cluster"] == c]
    ax.scatter(
        sub["media_cobertura"], sub["tendencia_cobertura"],
        s=12, alpha=0.6, linewidths=0, color=cores_cluster[c],
        label=f"{c} – {nomes_cluster[c]} (n={len(sub)})",
    )

ax.axhline(0, color=COR_EIXO, linewidth=1)
ax.set_xlabel("media_cobertura (%)", fontsize=10)
ax.set_ylabel("tendencia_cobertura (p.p./ano)", fontsize=10)
ax.set_title("Clusters de municípios por perfil de cobertura vacinal (2014–2023)", fontsize=12)
ax.legend(frameon=False, fontsize=9, loc="lower left")
ax.grid(axis="y", color=COR_GRADE, linewidth=0.8, zorder=0)
ax.set_axisbelow(True)
for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)
for spine in ["left", "bottom"]:
    ax.spines[spine].set_color(COR_EIXO)

fig.tight_layout()
fig.savefig("clusters_municipios.png", dpi=150)
plt.show()